<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/04_fulltext_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 全文検索 / 日本語トークナイズ (ブログ記事)

このノートブックは **pytidb シリーズの第 4 回** です。ブログ記事を題材に、キーワード一致ベースの全文検索 (FTS) を扱います。

## 学習目標
- `FullTextField(fts_parser="MULTILINGUAL")` で FTS インデックス付き列を作る
- `table.search(QUERY, search_type="fulltext")` で検索
- `match_score` の見方、ベクトル検索との違い
- 言語の異なる記事に対する `MULTILINGUAL` パーサの挙動

前提: `00_quickstart.ipynb` を実行済み。


## 1. 依存ライブラリのインストールとTiDB Cloudクラスタの作成


In [ ]:
!pip install -q pytidb


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-fts")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 2. ブログ記事テーブルの作成

ブログ記事テーブルを作成します。

`title` と `body` を全文検索可能にします。`FullTextField(fts_parser="MULTILINGUAL")` は
日本語・中国語・英語などを自動的にトークナイズできるパーサです。


In [ ]:
from pytidb import TiDBClient
from pytidb.schema import Field, FullTextField, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="blog_demo",
    ensure_db=True,
)


class Post(TableModel):
    __tablename__ = "posts"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    language: str = Field(max_length=8)
    title: str = FullTextField(fts_parser="MULTILINGUAL")
    body: str = FullTextField(fts_parser="MULTILINGUAL")


table = db.create_table(schema=Post, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 3. サンプル記事の投入

日本語 6 本 + 英語 3 本の小さなデータセットを投入します。


In [ ]:
POSTS = [
    ("ja", "分散トランザクションの基礎",        "TiDB の分散トランザクションは ACID で動作します。コミット時に TSO から一意のタイムスタンプを取得し、Percolator を参考にした 2 フェーズコミットで複数ノードにまたがる整合性を実現します。"),
    ("ja", "ベクトル検索を始めよう",            "ベクトル検索は意味的な近さでドキュメントを取り出す技術です。TiDB は VECTOR 型と HNSW インデックスを提供します。"),
    ("ja", "全文検索の使いどころ",              "キーワードが一致する文書を素早く取り出したい時は全文検索が向きます。商品カタログや FAQ のような用途で特に有効です。"),
    ("ja", "PyTiDB でレシピ RAG を作る",        "レシピデータをベクトル化し、ユーザーの質問に近いレシピを引っ張り出して LLM に渡すと、簡単に RAG アプリが作れます。"),
    ("ja", "HTAP で分析と運用を同じデータで",   "TiDB は行指向の TiKV と列指向の TiFlash を併用し、トランザクションと分析クエリを同じクラスタで実行できます。"),
    ("ja", "日本語トークナイザの違いを比べる",  "MULTILINGUAL パーサは漢字・ひらがな・カタカナを自動で区切ってくれます。英数字混じりでも自然に働きます。"),
    ("en", "Hybrid search explained",          "Hybrid search combines keyword based full-text matching with vector similarity to capture both exact and semantic relevance."),
    ("en", "Why use HTAP databases",           "HTAP systems let you run transactional and analytical workloads on the same cluster without ETL, making real-time dashboards easier."),
    ("en", "Intro to RAG for developers",      "Retrieval Augmented Generation improves LLM answers by grounding them on retrieved context from a knowledge base."),
]

table.bulk_insert([
    Post(id=i + 1, language=lang, title=t, body=b)
    for i, (lang, t, b) in enumerate(POSTS)
])
print(f"投入完了: {table.rows()} 件")


## 4. 全文検索を実行

`search_type="fulltext"` を指定し、検索対象のテキスト列を `.text_column()` で指定します。
スコアは `match_score`: 大きいほど一致度が高くなります。


In [ ]:
QUERY = "分散 トランザクション"
results = (
    table.search(QUERY, search_type="fulltext")
    .text_column("body")
    .limit(5)
    .to_pydantic()
)

print(f"Query: {QUERY}\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r.hit.language}] {r.hit.title}")
    print(f"    match_score={r.match_score:.4f}")


## 5. フィルタと組み合わせる

ベクトル検索同様、メソッドチェーンで言語別に絞る、タイトル列で検索するなど、他の列を利用したフィルタを組み合わせることができます。


In [ ]:
# 日本語の記事から "ベクトル" を検索
results = (
    table.search("ベクトル", search_type="fulltext")
    .text_column("body")
    .filter({"language": "ja"})
    .limit(3)
    .to_pydantic()
)
print("[日本語, body に 'ベクトル']")
for r in results:
    print(f"  {r.match_score:.3f}  {r.hit.title}")

# title 列で検索
print("\n[title 列で 'search' を検索]")
results = (
    table.search("search", search_type="fulltext")
    .text_column("title")
    .limit(5)
    .to_pydantic()
)
for r in results:
    print(f"  {r.match_score:.3f}  [{r.hit.language}] {r.hit.title}")


## 6. 全文検索 vs ベクトル検索

全文検索はキーワードの正確な一致に強く、ベクトル検索は意味の近さに強い、という違いがあります。
両方の良いとこ取りをするのが次回 (`05_hybrid_search.ipynb`) のハイブリッド検索です。


## 次のステップ
- `05_hybrid_search.ipynb` : ベクトル + 全文のハイブリッド検索

## チャレンジ
- 英語のクエリ (`"retrieval"`) で日本語記事がどう扱われるか試す
- 表記ゆれ (`"トランザクション" vs "トランザクシヨン"`) が match_score にどう影響するか見る
- `fts_parser="STANDARD"` に変えて挙動の違いを確認 (テーブルの再作成が必要)
